# Trabalho Final — Modelagem Funcional Geral (Etapa 3)

## Detecção de Uso de EPI em Ambiente Simples

**Disciplina:** ESZA019 — Visão Computacional — UFABC 2026.2
**Professor:** Celso Setsuo Kurashima
**Equipe:** Ctrl+C, Ctrl+V e Fé

### Integrantes da equipe
- Lucas Rodrigues Teixeira — RA 11202131394
- Pedro Henrique Garcez Silva — RA 11202130642
- Roberto Sene Azevedo — RA 11202020360

**Entregável:** Etapa 3 — Modelagem Funcional Geral (esboço)
**Data de publicação:** 15/07/2026

> **Escopo deste documento.** Esta é a entrega da **Etapa 3** do Trabalho Final: a **Modelagem
> Funcional Geral** do sistema. Trata-se de um documento de **projeto** (*design*), que define
> **requisitos técnicos, especificações, funções e funcionalidades, diagramas de blocos de
> implementação, o método de calibração e o método de avaliação funcional**. Ainda **não** contém
> resultados experimentais nem os testes com voluntários — estes serão produzidos nas etapas
> seguintes (Desenvolvimento, Etapa 4, e Testes, Etapas 5–7).


## Sumário

1. [Introdução, contexto e cenário de aplicação](#1-introdução-contexto-e-cenário-de-aplicação)
2. [Objetivo da modelagem funcional](#2-objetivo-da-modelagem-funcional)
3. [Requisitos técnicos e especificações](#3-requisitos-técnicos-e-especificações)
   - 3.1 [Delimitação do escopo — "ambiente simples"](#31-delimitação-do-escopo--ambiente-simples)
   - 3.2 [Requisitos funcionais (RF)](#32-requisitos-funcionais-rf)
   - 3.3 [Requisitos não funcionais (RNF)](#33-requisitos-não-funcionais-rnf)
   - 3.4 [Especificações de hardware e software](#34-especificações-de-hardware-e-software)
4. [Funções e funcionalidades do sistema](#4-funções-e-funcionalidades-do-sistema)
5. [Arquitetura e diagramas de blocos de implementação](#5-arquitetura-e-diagramas-de-blocos-de-implementação)
   - 5.1 [Diagrama de blocos geral (pipeline)](#51-diagrama-de-blocos-geral-pipeline)
   - 5.2 [Detalhamento dos blocos](#52-detalhamento-dos-blocos)
   - 5.3 [Fluxograma da lógica de decisão](#53-fluxograma-da-lógica-de-decisão)
   - 5.4 [Esboço de implementação (protótipo)](#54-esboço-de-implementação-protótipo)
6. [Método de calibração](#6-método-de-calibração)
7. [Método de avaliação funcional](#7-método-de-avaliação-funcional)
8. [Cronograma e próximas etapas](#8-cronograma-e-próximas-etapas)
9. [Declaração de uso de Inteligência Artificial Generativa](#9-declaração-de-uso-de-inteligência-artificial-generativa)
10. [Referências](#10-referências)


## 1. Introdução, contexto e cenário de aplicação

Equipamentos de Proteção Individual (**EPI**) — capacete, colete refletivo (alta visibilidade), óculos
de proteção, luvas — são a primeira barreira contra acidentes em canteiros de obra, oficinas e
laboratórios. A fiscalização do **uso correto** desses equipamentos, porém, ainda é feita quase sempre
de forma **manual e intermitente** por um responsável de segurança, o que é sujeito a falhas humanas,
não cobre todos os momentos e não gera registro objetivo.

A partir da fase de empatia do Trabalho Final (Etapas 1 e 2), a equipe definiu como **cenário de
aplicação** o **monitoramento automático do uso de EPI na entrada de uma área controlada** (por exemplo,
o acesso a uma oficina ou laboratório): uma câmera posicionada na entrada verifica, em **tempo real**, se
a pessoa que se aproxima está usando os EPIs obrigatórios e sinaliza **conforme / não conforme**.

Para manter o projeto **específico e bem delimitado** — conforme recomenda o item 8 do manual ("menos é
mais, desde que seja preciso") — o sistema é modelado para um **ambiente simples**: uma pessoa por vez,
iluminação controlada, fundo relativamente neutro e câmera fixa. Sobre esse cenário, a equipe adota a
trilha de **Deep Learning (CNNs)**, treinando/adaptando um **detector de objetos** para localizar, no
mesmo quadro, a **pessoa** e os **EPIs** (capacete e colete) e decidir a conformidade.

> **Observação sobre as entrevistas empáticas.** As entrevistas que fundamentam a escolha do tema, sua
> justificativa e originalidade foram conduzidas e relatadas na **Etapa 2 (Tema do Trabalho)**. Este
> documento de Etapa 3 concentra-se na **modelagem funcional** e apenas as referencia; os registros das
> entrevistas permanecem no entregável da Etapa 2.


## 2. Objetivo da modelagem funcional

Esta etapa tem por objetivo **especificar de forma completa como o sistema funcionará**, antes de sua
implementação. Concretamente, o documento define:

- os **requisitos técnicos** e as **especificações** do sistema (o que ele deve fazer e sob quais
  condições);
- as **funções e funcionalidades** (decomposição do sistema em capacidades);
- os **diagramas de blocos de implementação** (como os módulos se conectam e qual o fluxo dos dados);
- o **método de calibração** da câmera (requisito obrigatório do trabalho);
- o **método de avaliação funcional** (as métricas objetivas que comprovarão a robustez da solução).

A **trilha técnica escolhida** (Requisito B do manual) é a de **Deep Learning (CNNs)**: um detector de
objetos de estágio único (família **YOLO**, ou alternativa leve como **MobileNet-SSD**) é treinado/
adaptado para as classes de interesse (**pessoa**, **capacete**, **colete**), e uma etapa de **associação
espacial** e **decisão** transforma as detecções em um veredito de conformidade por pessoa, estabilizado
por **rastreamento** temporal.


## 3. Requisitos técnicos e especificações

### 3.1 Delimitação do escopo — "ambiente simples"

O sistema é projetado para as seguintes condições controladas, que caracterizam o "ambiente simples":

| Condição | Especificação de projeto |
|----------|--------------------------|
| Número de pessoas | **Uma pessoa por vez** no campo de visão (fila/eclusa de entrada). |
| Iluminação | Interna, **difusa e estável** (sem contraluz forte nem sombras duras). |
| Fundo | Relativamente **neutro e estático** (parede/piso de cor uniforme). |
| Câmera | **Fixa** (tripé/suporte), previamente **calibrada**. |
| Distância | Pessoa a ~1,5–3 m da câmera, aproximadamente frontal. |
| EPIs alvo | **Capacete** e **colete de alta visibilidade** (classes do detector). |

**Justificativa da delimitação.** Restringir a um único usuário, iluminação e fundo controlados **reduz a
variabilidade** que o modelo precisa aprender, permitindo bons resultados com um **detector leve** e um
**conjunto de treino modesto** (poucas centenas a alguns milhares de imagens), adequado ao prazo e ao
objetivo pedagógico do trabalho. "Menos é mais, desde que seja preciso" (item 8 do manual).


### 3.2 Requisitos funcionais (RF)

| ID | Requisito funcional |
|----|---------------------|
| RF-01 | Capturar vídeo em **tempo real** de uma webcam/câmera USB. |
| RF-02 | Aplicar a **calibração** previamente obtida para corrigir a distorção de cada quadro. |
| RF-03 | Executar **inferência de um detector CNN** que localize *pessoa*, *capacete* e *colete* em cada quadro. |
| RF-04 | Aplicar **NMS** (*non-maximum suppression*) e um **limiar de confiança** às detecções. |
| RF-05 | **Associar** cada EPI detectado à pessoa correspondente (por contenção/IoU das *bounding boxes*). |
| RF-06 | **Classificar** o estado da pessoa como **CONFORME** (pessoa + capacete + colete) ou **NÃO CONFORME**. |
| RF-07 | **Rastrear** a pessoa entre quadros para estabilizar a decisão e tratar oclusões parciais. |
| RF-08 | **Exibir** o resultado sobreposto ao vídeo (caixas, rótulos, confiança) e emitir **alerta** visual. |
| RF-09 | **Registrar** métricas de execução (FPS/latência) e permitir a coleta de dados para validação. |


### 3.3 Requisitos não funcionais (RNF)

| ID | Requisito não funcional | Meta de projeto |
|----|--------------------------|-----------------|
| RNF-01 | **Desempenho em tempo real** | ≥ 15 FPS (modelo leve; GPU quando disponível). |
| RNF-02 | **Latência** de inferência por quadro | ≤ 66 ms/quadro. |
| RNF-03 | **Robustez temporal** | Decisão confirmada por *N* quadros consecutivos (evita "piscar"). |
| RNF-04 | **Portabilidade** | Rodar em Linux com Python 3, OpenCV e o *runtime* do modelo (PyTorch/ONNX). |
| RNF-05 | **Reprodutibilidade** | *Pesos*, conjunto de dados e sementes versionados; treino documentado. |
| RNF-06 | **Usabilidade** | Interface visual clara (verde = conforme, vermelho = não conforme). |


### 3.4 Especificações de hardware e software

**Hardware.**
- Webcam USB (≥ 640×480, 30 FPS) ou câmera do notebook.
- Tripé/suporte para manter a câmera **fixa** (a calibração exige que a câmera não se mova).
- Tabuleiro de xadrez impresso (padrão de calibração, cantos internos conhecidos).
- Computador com **GPU** recomendada para treino; a inferência em tempo real roda em **CPU** com um
  modelo leve (YOLO *nano*/*small* ou MobileNet-SSD).

**Software.**
- **Python 3**, **OpenCV 4.x**, **NumPy**.
- *Framework* de *deep learning*: **PyTorch** (com **Ultralytics YOLO**) ou execução via **ONNX Runtime**;
  leitura do modelo também possível pelo módulo **`cv2.dnn`** do OpenCV.
- Reaproveitamento direto dos **Laboratórios 4 e 5** (calibração de câmera) da própria equipe.
- Módulos previstos: `aquisicao`, `calibracao`, `detector_cnn`, `associacao`, `decisao`, `rastreamento`,
  `interface`, `metricas`.


## 4. Funções e funcionalidades do sistema

O sistema é decomposto nas seguintes **funções** (capacidades internas) e **funcionalidades**
(o que o usuário percebe):

**Funções (internas).**
1. **Aquisição de vídeo** — leitura contínua de quadros da câmera.
2. **Correção geométrica** — *undistort* de cada quadro com os parâmetros de calibração.
3. **Inferência CNN (detector)** — passagem do quadro pela rede (**YOLO**/MobileNet-SSD), retornando
   *bounding boxes*, classes (*pessoa*, *capacete*, *colete*) e confianças.
4. **Pós-processamento** — filtragem por **confiança** e **NMS** para remover caixas redundantes.
5. **Associação espacial** — vínculo de cada EPI à pessoa correspondente por **contenção/IoU** das caixas
   (o capacete deve estar na região da cabeça; o colete, no tronco).
6. **Rastreamento** — associação da pessoa entre quadros (rastreador leve tipo **SORT/ByteTrack** ou
   `cv2` *tracker*), com histórico para **decisão temporal** e tratamento de **oclusões**.
7. **Decisão e classificação** — regra lógica (pessoa ∧ capacete ∧ colete) → **CONFORME / NÃO CONFORME**,
   confirmada por *N* quadros.
8. **Medição de desempenho** — cálculo de **FPS/latência** e exportação de dados para validação.

**Funcionalidades (percebidas pelo usuário).**
- Vídeo ao vivo com **caixas** de pessoa e EPIs, **classe** e **confiança** de cada detecção.
- **Status global** destacado (faixa verde/vermelha) e **alerta** quando não conforme.
- **Contador de FPS** na tela e opção de **gravar** a sessão para análise posterior.


## 5. Arquitetura e diagramas de blocos de implementação

### 5.1 Diagrama de blocos geral (pipeline)

O fluxo de dados percorre os blocos abaixo, do sensor à decisão:

```
   [ Câmera USB ]
        |
        v
+------------------+     +---------------------+     +--------------------------+
|  (1) Aquisição   | --> |  (2) Calibração /   | --> |  (3) Inferência CNN      |
|  de vídeo (BGR)  |     |  undistort (K,dist) |     |  (YOLO / MobileNet-SSD)  |
+------------------+     +---------------------+     +------------+-------------+
                                                                  | deteccoes (box,classe,conf)
                                                                  v
                                                   +--------------------------------+
                                                   |  (4) Pos-proc: confianca + NMS |
                                                   +---------------+----------------+
                                                                   |
                                                                   v
                                        +----------------------------------------------+
                                        |  (5) Associacao EPI <-> pessoa (IoU/contencao)|
                                        +----------------------+-----------------------+
                                                               |
                                                               v
                                             +----------------------------------+
                                             | (6) Rastreamento + decisao        |
                                             | temporal (N quadros)              |
                                             +-----------------+----------------+
                                                               v
                                         +-------------------------------------+
                                         | (7) Saida: caixas + status +        |
                                         |     alerta + metricas (FPS)         |
                                         +-------------------------------------+
```

Em paralelo (fora de tempo real) existe o **fluxo de treino** do detector, executado **uma vez** na
Etapa 4:

```
[ Dataset de EPI ] -> [ Anotacao/Split ] -> [ Treino da CNN ] -> [ Validacao (mAP) ] -> [ Pesos .pt/.onnx ]
                                                                                              |
                                                                        usado pelo bloco (3) --+
```


### 5.2 Detalhamento dos blocos

| Bloco | Entrada | Processamento | Saída |
|-------|---------|---------------|-------|
| (1) Aquisição | Câmera USB | `cv2.VideoCapture` lê o quadro | Quadro BGR |
| (2) Calibração | Quadro BGR + `K`, `dist` | `cv2.undistort` (parâmetros do Lab 4) | Quadro sem distorção |
| (3) Inferência CNN | Quadro corrigido | *Forward* da rede (YOLO/SSD): *bounding boxes* + classe + confiança | Lista de detecções |
| (4) Pós-proc. | Detecções | Limiar de confiança + **NMS** | Detecções filtradas |
| (5) Associação | Caixas de pessoa e de EPI | **IoU/contenção** para vincular EPI à pessoa | Pessoa com EPIs atribuídos |
| (6) Rastreamento/decisão | Pessoas por quadro | Rastreio (SORT/ByteTrack) + confirmação em *N* quadros | Estado estável por pessoa |
| (7) Saída | Estado + evidências | Desenho de caixas/rótulos, alerta, FPS | Vídeo anotado + *log* |

**Nota de projeto (ordem calibração × processamento).** Conforme consolidado no Laboratório 1, qualquer
processamento sobre a imagem ocorre **depois da captura/leitura do quadro** e **antes da exibição ou
gravação**. Por isso a **calibração (undistort)** é o **primeiro** processamento do *pipeline*: a CNN e
todos os blocos seguintes trabalham sobre a imagem já corrigida.


### 5.3 Fluxograma da lógica de decisão

```
             +----------------------------+
             |  Pessoa detectada (CNN)?    |
             +-------------+--------------+
                           | nao
                 +---------+---------+
                 |                   v
                 | sim         [ aguarda / sem status ]
                 v
      +-------------------------------+
      | Capacete associado a pessoa?  |
      +------+-------------+----------+
             | sim         | nao
             v             v
   +----------------------+   [ NAO CONFORME: falta capacete ]
   | Colete associado a   |
   | pessoa?              |
   +----+------------+----+
        | sim        | nao
        v            v
 [ CONFORME ]   [ NAO CONFORME: falta colete ]
        |
        v
 confirma em N quadros consecutivos -> emite status estavel + alerta se NAO CONFORME
```


### 5.4 Esboço de implementação (protótipo)

Os trechos abaixo são um **esboço** (não a versão final) que ilustra como cada bloco será implementado com
**deep learning**. Servem para fixar as interfaces entre os módulos; o modelo, os *pesos* e os limiares
serão **treinados/ajustados** na Etapa 4.


In [ ]:
import cv2
import numpy as np

# --- Bloco (2): calibracao previamente obtida (Lab 4). Valores de exemplo. ---
K = np.array([[692.56, 0., 320.24],
              [0., 689.08, 247.95],
              [0., 0., 1.]])
dist = np.array([[-0.0325, 0.4864, 0.0025, -0.0097, -1.2327]])

# --- Bloco (3): detector CNN (YOLO via Ultralytics). Pesos a treinar na Etapa 4. ---
# from ultralytics import YOLO
# modelo = YOLO("epi_yolo.pt")   # classes: 0=pessoa, 1=capacete, 2=colete
CONF_MIN = 0.35                  # limiar de confianca (Bloco 4)

def iou(a, b):
    """Interseccao sobre uniao entre duas caixas (x, y, w, h)."""
    ax, ay, aw, ah = a; bx, by, bw, bh = b
    x1, y1 = max(ax, bx), max(ay, by)
    x2, y2 = min(ax+aw, bx+bw), min(ay+ah, by+bh)
    inter = max(0, x2-x1) * max(0, y2-y1)
    union = aw*ah + bw*bh - inter
    return inter/union if union > 0 else 0.0


In [ ]:
def decidir(deteccoes):
    """Blocos (5)+(6): associa EPIs a cada pessoa e decide conformidade.
    'deteccoes' = lista de dicts {classe, box, conf} ja filtrados por confianca/NMS."""
    pessoas = [d for d in deteccoes if d["classe"] == "pessoa"]
    capacetes = [d for d in deteccoes if d["classe"] == "capacete"]
    coletes  = [d for d in deteccoes if d["classe"] == "colete"]

    resultados = []
    for p in pessoas:
        # um EPI "pertence" a pessoa se sua caixa esta contida/sobreposta a dela.
        tem_capacete = any(iou(p["box"], c["box"]) > 0.05 for c in capacetes)
        tem_colete  = any(iou(p["box"], v["box"]) > 0.05 for v in coletes)
        conforme = tem_capacete and tem_colete
        resultados.append({"box": p["box"], "capacete": tem_capacete,
                            "colete": tem_colete, "conforme": conforme})
    return resultados

# --- Laco principal (esboco): aquisicao -> undistort -> CNN -> decisao -> saida ---
# cap = cv2.VideoCapture(0)
# while True:
#     ok, frame = cap.read()
#     if not ok: break
#     frame = cv2.undistort(frame, K, dist)                         # (2)
#     pred = modelo(frame, conf=CONF_MIN)[0]                        # (3)+(4): infer + NMS
#     deteccoes = para_lista(pred)                                  # converte saida da rede
#     for r in decidir(deteccoes):                                  # (5)+(6)
#         x, y, w, h = r["box"]
#         cor = (0, 200, 0) if r["conforme"] else (0, 0, 255)
#         cv2.rectangle(frame, (x, y), (x+w, y+h), cor, 2)          # (7)
#     cv2.imshow("Deteccao de EPI (CNN)", frame)
#     if cv2.waitKey(1) == ord('q'): break
# cap.release(); cv2.destroyAllWindows()


## 6. Método de calibração

A **calibração de câmera é obrigatória** (Requisito 3.A do manual) e reaproveita diretamente o método
validado pela equipe no **Laboratório 4**. O objetivo é **corrigir a distorção** das lentes antes da
inferência, garantindo que as *bounding boxes* e a associação espacial não sejam deturpadas pela
curvatura das bordas.

**Modelo.** A projeção segue o modelo *pinhole* $s\,[u\;v\;1]^\top = K\,[R\mid t]\,[X\;Y\;Z\;1]^\top$, com

$$K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}, \qquad
\texttt{dist} = (k_1, k_2, p_1, p_2, k_3),$$

onde $K$ reúne os **intrínsecos** (distância focal e ponto principal), $[R\mid t]$ os **extrínsecos**
(pose) e `dist` os coeficientes de **distorção radial** ($k_i$) e **tangencial** ($p_i$).

**Procedimento.**
1. Imprimir o **tabuleiro de xadrez** (cantos internos conhecidos, p. ex. $6\times8$).
2. Capturar de **10 a 15 imagens** do tabuleiro em poses variadas com a câmera do sistema
   (programa `L4_chess.py` do Lab 4).
3. Detectar e refinar os cantos (`findChessboardCorners` + `cornerSubPix`) e estimar os parâmetros com
   `cv2.calibrateCamera`, obtendo `K`, `dist`, e os pares $(R,t)$ por imagem.
4. **Corrigir a distorção** de cada quadro em tempo real com `cv2.getOptimalNewCameraMatrix` +
   `cv2.undistort` (ou `cv2.remap`).

**Demonstração do erro de reprojeção (exigida pelo manual).** A qualidade da calibração é medida pelo
**erro de reprojeção**, a distância média (em pixels) entre os cantos detectados e os cantos
**reprojetados** com os parâmetros estimados:

$$\text{erro} = \frac{1}{N}\sum_{i=1}^{N} \big\lVert \, \mathbf{p}_i^{\text{obs}} - \pi(K, \texttt{dist}, R_i, t_i, \mathbf{P}_i) \, \big\rVert_2$$

calculada com `cv2.projectPoints` e `cv2.norm`; um RMS **abaixo de ~1 px** indica calibração adequada. O
valor obtido será reportado no Relatório Técnico (Semana 11).


In [ ]:
# Método padrão para a DEMONSTRAÇÃO DO ERRO DE REPROJEÇÃO (a ser reportado com dados reais).
def erro_reprojecao(objpoints, imgpoints, rvecs, tvecs, K, dist):
    erro_total, n = 0.0, 0
    for i in range(len(objpoints)):
        proj, _ = cv2.projectPoints(objpoints[i], rvecs[i], tvecs[i], K, dist)
        erro = cv2.norm(imgpoints[i], proj, cv2.NORM_L2) / len(proj)
        erro_total += erro
        n += 1
    return erro_total / n   # erro medio de reprojeicao (pixels)


## 7. Método de avaliação funcional

A validação segue o item 4 do manual ("provar que o sistema é robusto") por meio de **métricas
objetivas**. Todas serão medidas na Etapa 4 (protótipo) e nos testes com voluntários (Etapas 5–7); aqui
definimos **como** cada uma será obtida.

**7.1 Desempenho — Latência / FPS.** Mede-se o tempo por quadro (aquisição + *undistort* + inferência +
decisão) e a taxa média: $\text{FPS} = 1 / \Delta t_{\text{quadro}}$. Meta: **≥ 15 FPS** e latência
**≤ 66 ms/quadro** (RNF-01/02).

**7.2 Detecção — mAP.** Como o núcleo é um detector, reporta-se a **mAP** (*mean Average Precision*, em
IoU 0,5) no conjunto de validação, métrica-padrão para detecção de objetos, por classe (*pessoa*,
*capacete*, *colete*) e média.

**7.3 Classificação (decisão de conformidade) — Matriz de confusão.** A decisão final por pessoa/quadro é
binária: **CONFORME** × **NÃO CONFORME**. Comparando a saída com o *ground truth* anotado, constrói-se:

|  | **Predito: Conforme** | **Predito: Não conforme** |
|---|---|---|
| **Real: Conforme** | VP | FN |
| **Real: Não conforme** | FP | VN |

E derivam-se as métricas:

$$\text{Precisão} = \frac{VP}{VP+FP}, \qquad \text{Revocação (Recall)} = \frac{VP}{VP+FN}, \qquad
F1 = 2\cdot\frac{\text{Precisão}\cdot\text{Revocação}}{\text{Precisão}+\text{Revocação}}$$

> No contexto de **segurança**, prioriza-se **alta revocação** para a classe "NÃO CONFORME" (é preferível
> um alarme falso a **deixar passar** alguém sem EPI). Esse critério guiará o ajuste do limiar de confiança.

**7.4 Erro espacial (apoio).** Como a câmera é calibrada, pode-se reportar o **erro médio em pixels** na
localização das caixas, útil para diagnosticar falhas de enquadramento/associação.

**7.5 Protocolo de teste com voluntários (Etapas 5–7).**
1. **Roteiro de tarefas:** o voluntário entra na área (a) com todos os EPIs, (b) sem capacete, (c) sem
   colete e (d) sem nenhum — cobrindo as quatro situações.
2. **Filmagem** (celular) da interação, conforme o manual.
3. **Anotação do *ground truth*** quadro a quadro e cálculo das métricas de 7.1–7.4.
4. **Coleta de feedback:** dificuldades, erros do sistema e sugestões de melhoria.

**7.6 Modelo de relato dos resultados (a preencher com dados reais).**

| Métrica | Meta | Valor obtido |
|---------|------|--------------|
| FPS médio | ≥ 15 | _a preencher_ |
| Latência média (ms) | ≤ 66 | _a preencher_ |
| mAP@0,5 (média) | — | _a preencher_ |
| Precisão (decisão) | — | _a preencher_ |
| Revocação (NÃO CONFORME) | alta | _a preencher_ |
| F1-Score | — | _a preencher_ |
| Erro de reprojeção (calibração) | < 1 px | _a preencher_ |

> Os valores acima são **campos de preenchimento**: serão obtidos **experimentalmente** pela equipe. Nada
> nesta seção representa resultados medidos.


## 8. Cronograma e próximas etapas

Esta entrega corresponde à **Etapa 3 — Modelagem Funcional Geral**. As etapas subsequentes, conforme o
item 9 do manual, são:

| Etapa | Atividade | Situação |
|-------|-----------|----------|
| 1 | Entrevistas empáticas | Concluída (Etapa 2) |
| 2 | Contexto e cenário de aplicação | Concluída (Etapa 2) |
| **3** | **Modelagem funcional geral (este documento)** | **Entregue — 15/07/2026** |
| 4 | Desenvolvimento do projeto (dataset, treino da CNN, protótipo) | A fazer |
| 5 | Elaboração do roteiro de teste com voluntários | A fazer |
| 6–7 | Seminários: testes com voluntários e análise de resultados | A fazer |
| 8 | Simpósio: artigo e apresentação | A fazer |


## 9. Declaração de uso de Inteligência Artificial Generativa

Em atendimento à **Portaria CNPq nº 2664/2026**, a equipe declara o uso de assistente de IA generativa
(modelo de linguagem da Anthropic — Claude) como **ferramenta de apoio** na preparação deste documento.

- **Finalidade:** organização e estruturação das seções, apoio à redação e revisão do texto, comentários
  do esboço de código e formatação do notebook.
- **Autoria e responsabilidade:** a **concepção do tema, o cenário de aplicação, os requisitos e as
  decisões de arquitetura** são da equipe, derivados das entrevistas empáticas e das aulas da disciplina.
  A ferramenta de IA **não gerou dados nem resultados experimentais** — nenhum resultado, métrica ou
  entrevista foi fabricado. A equipe revisou e validou todo o conteúdo e é **integralmente responsável**
  pelo material final.


## 10. Referências

1. REDMON, J. et al. *You Only Look Once: Unified, Real-Time Object Detection*. CVPR, 2016.
2. JOCHER, G. et al. *Ultralytics YOLO* (documentação). Disponível em: https://docs.ultralytics.com/
3. HOWARD, A. et al. *MobileNets: Efficient CNNs for Mobile Vision Applications*. 2017.
4. OpenCV — *Deep Neural Network module (`cv2.dnn`)*. Disponível em:
   https://docs.opencv.org/4.x/d6/d0f/group__dnn.html
5. OpenCV — *Camera Calibration*. Disponível em:
   https://docs.opencv.org/4.x/dc/dbb/tutorial_py_calibration.html
6. LearnOpenCV — *Camera Calibration using OpenCV*. Disponível em:
   https://learnopencv.com/camera-calibration-using-opencv/
7. CNPq — *Portaria nº 2664/2026* (diretrizes de integridade e uso de IAG). Disponível em:
   http://memoria2.cnpq.br/web/guest/view/-/journal_content/56_INSTANCE_0oED/10157/23142775
